In [28]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import array_contains, col, explode

spark = (SparkSession.builder
         .appName("filter-data")
         .master("spark://spark-master:7077")
         .config("spark.executor.memory", "512m")
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

In [29]:
df = (spark.read.format("csv")
      .option("header", "true")
      .option("nullValue", "null")
      .option("dateFormat", "LLLL d, y")
      .load("../data/netflix_titles.csv"))

In [30]:
df.printSchema()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)



In [3]:
filtered_df = df.filter(col("release_year") > 2020)
filtered_df.show()

+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|            director|                cast|             country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|     s2|TV Show|       Blood & Water|                null|Ama Qamata, Khosi...|        South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglands|     Julien Leclercq|Sami Bouajila, Tr...|                null|September 24, 2021|        2021| TV-MA| 1 Season|Crime TV Shows, I...|To protect his fa...|
|     s4|TV Show|Jailbirds New Orl.

In [4]:
filtered_df = (
    df.filter(
        (col("country") == "United States") & (col("release_year") > 2020)
    )
)

filtered_df.show()

+-------+-------+--------------------+--------------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|            director|                cast|      country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+--------------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|    s10|  Movie|        The Starling|      Theodore Melfi|Melissa McCarthy,...|United States|September 24, 2021|        2021| PG-13|  104 min|    Comedies, Dramas|A woman adjusting...|
|    s16|TV Show|   Dear White People|                null|Logan Browning, B...|United States|September 22, 2021|        2021| TV-MA|4 Seasons|TV Comedies, TV D...|"Students of colo...|
|    s41|TV Show|He-Man and the Ma...|                null|Yuri Lowent

In [5]:
filtered_df = (
    df.filter(
        col("country")
        .isin([
            "United States",
            "United Kingdom",
            "India"
        ])
    )
)

filtered_df.show(3)

+-------+-------+--------------------+---------------+--------------------+--------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|       director|                cast|       country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+---------------+--------------------+--------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|Kirsten Johnson|                null| United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s5|TV Show|        Kota Factory|           null|Mayur More, Jiten...|         India|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|In a city of coac...|
|     s9|TV Show|The Great British...|Andy Devonshire|Mel Giedroyc, Sue...|United Kingdom|

### Filtering on string

In [6]:
filtered_df = df.filter(col("listed_in").like("%Crime%"))
filtered_df.show()

+-------+-------+--------------------+--------------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|            director|                cast|      country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+--------------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|     s3|TV Show|           Ganglands|     Julien Leclercq|Sami Bouajila, Tr...|         null|September 24, 2021|        2021| TV-MA| 1 Season|Crime TV Shows, I...|To protect his fa...|
|    s11|TV Show|Vendetta: Truth, ...|                null|                null|         null|September 24, 2021|        2021| TV-MA| 1 Season|Crime TV Shows, D...|"Sicily boasts a ...|
|    s12|TV Show|    Bangkok Breaking|   Kongkiat Komesiri|Sukollawat 

In [7]:
filtered_df = df.filter(col("listed_in").rlike("(Crime|Thrillers)"))
filtered_df.show()

+-------+-------+--------------------+-----------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|         director|                cast|             country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+-----------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|     s3|TV Show|           Ganglands|  Julien Leclercq|Sami Bouajila, Tr...|                null|September 24, 2021|        2021| TV-MA| 1 Season|Crime TV Shows, I...|To protect his fa...|
|    s11|TV Show|Vendetta: Truth, ...|             null|                null|                null|September 24, 2021|        2021| TV-MA| 1 Season|Crime TV Shows, D...|"Sicily boasts a ...|
|    s12|TV Show|    Bangkok Breaking|Kongkiat Kom

### Filtering on Data Ranges

In [16]:
# filter the DataFrame based on a date range
filtered_df = df.filter((col("release_year") >= "2021") & (col("release_year") <= "2022"))

# display the filtered DataFrame
filtered_df.show()

AssertionError: 

In [32]:
SparkContext

NameError: name 'SparkContext' is not defined

In [37]:
# filter the DataFrame based on a date range
# filtered_df = df.filter((col("release_year").cast("int").between("2021","2021")))
filtered_df = df.withColumn("test", col("release_year").cast("int"))
filtered_df = filtered_df.filter((col("test").between("2019", "2020")))
# display the filtered DataFrame
filtered_df.show()

+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+----+
|show_id|   type|               title|            director|                cast|             country|        date_added|release_year|rating| duration|           listed_in|         description|test|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+----+
|     s1|  Movie|Dick Johnson Is Dead|     Kirsten Johnson|                null|       United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|2020|
|    s17|  Movie|Europe's Most Dan...|Pedro de Echave G...|                null|                null|September 22, 2021|        2020| TV-MA|   67 min|Documentaries, In...|Declassified docu...|2020|
|    s18|T

### Filter on Arrays

In [38]:
from pyspark.sql.functions import array_contains, col

In [39]:
df_recipes = (
    spark.read.format("parquet")
    .load("../data/recipes.parquet")
)

In [41]:
df_recipes.printSchema()

root
 |-- RecipeId: double (nullable = true)
 |-- Name: string (nullable = true)
 |-- AuthorId: integer (nullable = true)
 |-- AuthorName: string (nullable = true)
 |-- CookTime: string (nullable = true)
 |-- PrepTime: string (nullable = true)
 |-- TotalTime: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- RecipeCategory: string (nullable = true)
 |-- Keywords: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- RecipeIngredientQuantities: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- RecipeIngredientParts: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- AggregatedRating: double (nullable = true)
 |-- ReviewCount: integer (nullable = true)
 |-- Calories: double (nullable = true)
 |-- FatContent: double (nullable = true)
 |-- SaturatedFatContent: double (nullable = true)
 |-- CholesterolContent: double (nullable = true)
 |-- SodiumContent: double (nullable = true)
 |-- Carbohydr

In [44]:
df_recipes.show(1, truncate=False)

+--------+------------------------------+----------+------------+--------+--------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+-----------+-------------------------------------------+--------------------------------------------------------------------------------------+----------------+-----------+--------+----------+-------------------+------------------+-------------+-------------------+------------+------------+--------------+--------------+-----------+---------------------------------------------------------

In [40]:
filtered_df = df_recipes.filter(array_contains(col("RecipeIngredientParts"), "apple"))

In [46]:
filtered_df.show(2, truncate=False)

+--------+----------------------------+----------+-----------------+--------+--------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+-------------------------------------+--------------------------------+-------------------------------------------------------------------+----------------+-----------+--------+----------+-------------------+------------------+-------------+-------------------+------------+------------+--------------+--------------+-----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Filtering on map columns

In [47]:
df_nobel_prizes = (
    spark.read.format("json")
    .option("multiLine", "true")
    .load("../data/nobel_prizes.json")
)

In [48]:
df_nobel_prizes.printSchema()

root
 |-- category: string (nullable = true)
 |-- laureates: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- firstname: string (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- motivation: string (nullable = true)
 |    |    |-- share: string (nullable = true)
 |    |    |-- surname: string (nullable = true)
 |-- overallMotivation: string (nullable = true)
 |-- year: string (nullable = true)



In [51]:
df_nobel_prizes.show(2)

+---------+--------------------+-----------------+----+
| category|           laureates|overallMotivation|year|
+---------+--------------------+-----------------+----+
|chemistry|[{Carolyn, 1015, ...|             null|2022|
|economics|[{Ben, 1021, "for...|             null|2022|
+---------+--------------------+-----------------+----+
only showing top 2 rows



In [53]:
df_nobel_prizes_exploded = (
    df_nobel_prizes
    .withColumn("laureates", explode(col("laureates")))
    .select(
         col("category")
        ,col("year")
        ,col("overallMotivation")
        # ,col("laureates.firstname")
        ,col("laureates")
    )
)

df_nobel_prizes_exploded.show(2, truncate=False)

+---------+----+-----------------+--------------------------------------------------------------------------------------------------+
|category |year|overallMotivation|laureates                                                                                         |
+---------+----+-----------------+--------------------------------------------------------------------------------------------------+
|chemistry|2022|null             |{Carolyn, 1015, "for the development of click chemistry and bioorthogonal chemistry", 3, Bertozzi}|
|chemistry|2022|null             |{Morten, 1016, "for the development of click chemistry and bioorthogonal chemistry", 3, Meldal}   |
+---------+----+-----------------+--------------------------------------------------------------------------------------------------+
only showing top 2 rows



In [55]:
filtered_df = (
    df_nobel_prizes_exploded
    .filter(
        (col("laureates").getItem("firstname")=="Albert") & (col("laureates").getItem("surname")=="Einstein")
    )
)

In [58]:
filtered_df.show(2, truncate=False)

+--------+----+-----------------+---------------------------------------------------------------------------------------------------------------------------------------------+
|category|year|overallMotivation|laureates                                                                                                                                    |
+--------+----+-----------------+---------------------------------------------------------------------------------------------------------------------------------------------+
|physics |1921|null             |{Albert, 26, "for his services to Theoretical Physics, and especially for his discovery of the law of the photoelectric effect", 1, Einstein}|
+--------+----+-----------------+---------------------------------------------------------------------------------------------------------------------------------------------+



In [59]:
spark.stop()